# Integrated databases and tools for agentic toxicology

SOT 2027 CE course demo. Three tools, one agent loop, provenance on every line.

1. **Data**: a slice of the BioBricks toxicology knowledge graph (`biobricks-ai/ice`, the NICEATM Integrated Chemical Environment) plus PubChem for chemical identity.
2. **Model**: ToxTransformer, a public MIT-licensed 59.2M-parameter transformer, run on CPU.
3. **Agent**: a rule-based planner that chooses tools and refuses to state anything a tool did not return.

Everything here is public. Nothing needs an API key.

In [ ]:
from agentic_tox import kg, model, agent
print("endpoints this demo pins:")
for key, endpoint in model.ENDPOINTS.items():
    print(f"  {key:30s} index {endpoint.index:5d}  [{endpoint.catalog_source}]  {endpoint.title}")

## 1. Pull the data slice

BioBricks stores brick assets as content-addressed objects, so one Parquet table can
be fetched over anonymous HTTPS and checked against the MD5 the brick records. No
token, no 70 MB pull of the whole brick.

In [ ]:
listing = kg.brick_listing()
print(f"{len(listing)} tables in the brick; this demo reads {sorted(kg.TABLES)}")

cancer, source = kg.load_table("Cancer")
print(cancer.shape)
print(source.line())
cancer[cancer.CASRN == "131-57-7"][["Chemical.Name", "Assay", "Endpoint", "Response", "Species"]]

## 2. Resolve the chemical, then run the model

The model predicts on a structure, so where the structure came from is part of the
answer. The checkpoint is downloaded once (237 MB) and verified against the SHA-256
published in the upstream repository's own manifest.

In [ ]:
record, facts = kg.resolve_chemical("131-57-7")
for fact in facts:
    print("\n".join(fact.lines()))
record

In [ ]:
for fact in model.predict_panel(record["smiles"], model.MUTAGENICITY_ENDPOINTS):
    print("\n".join(fact.lines()))

## 3. The agent loop

The deterministic planner reads the question, picks the tools, and assembles an
answer in which every claim names its source. Swap in an LLM planner by setting
`AGENTIC_TOX_LLM=anthropic` (or `openai`) with the matching API key; the no-key path
is the tested default.

In [ ]:
result = agent.answer(
    "Is oxybenzone (CAS 131-57-7) likely mutagenic, and what does the evidence say?",
    planner="deterministic",
)
print(result.render())

## 4. The trace is machine-readable

The same answer as JSON: one record per fact, each with dataset digest or model
weights hash, URL and timestamp. This is what you would attach to a report, or feed
to a reviewer who wants to check the agent rather than trust it.

In [ ]:
import json
print(json.dumps(result.to_dict()["steps"][2]["facts"][-1], indent=2)[:1500])

## What this shows, and what it does not

**Shows**: public toxicology data is addressable and verifiable; a public model runs
on a laptop CPU in under a second per prediction; an agent can be made to cite
everything; measured data and prediction can be compared side by side.

**Does not show**: that the model is fit for a regulatory submission; that these
assay-level binary endpoints are an OECD 471 call; anything about metabolism,
exposure or dose. The ICE slice here is two tables out of 27.